In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser

from langchain_core.messages  import HumanMessage, SystemMessage, AnyMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

In [2]:
# langraph-cli

# langgraph dev

In [3]:
credential_paths= r"D:\common_credentials\.env"
load_dotenv(dotenv_path=credential_paths)
os.getenv("LANGSMITH_PROJECT")

'ragapplication'

In [4]:
## laod env vars
# os.environ["OPENAI_API_KEY"]= os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_TRACING"]=os.getenv("LANGSMITH_TRACING")
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT")
os.environ["LANGSMITH_ENDPOINT"]=os.getenv("LANGSMITH_ENDPOINT")
os.environ["OPENAI_API_KEY"]= "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"
os.environ["LANGCHAIN_PROJECT"]= os.getenv("LANGSMITH_ENDPOINT")

In [5]:
import yaml
with open("../../config.yml", "r") as yml_file:
    yml_config= yaml.safe_load(yml_file)
    yml_file.close()
print(yml_config['GroqModels']["Inferencing_Models"]['text_models'])

['distil-whisper-large-v3-en', 'gemma2-9b-it', 'llama-3.3-70b-versatile', 'llama-3.1-8b-instant', 'whisper-large-v3-turbo', 'deepseek-r1-distill-llama-70b']


In [6]:
llm = ChatGroq(model="llama-3.1-8b-instant")
output_parser= StrOutputParser()
basic_chain= llm | output_parser
basic_chain.invoke("Tell me a joke")

'A man walked into a library and asked the librarian, "Do you have any books on Pavlov\'s dogs and Schrödinger\'s cat?" \n\nThe librarian replied, "It rings a bell, but I\'m not sure if it\'s here or not."'

## structure output

In [43]:
from pydantic import BaseModel, Field
from typing import List, Any, TypedDict
from langchain_core.output_parsers import JsonOutputParser

class MobileReview(BaseModel):
    phone_model:str= Field(description="Name and model of phone")
    rating:float= Field(description= "Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

review_text = """
Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
stronger. Battery life's solid, lasts me all day no problem.

Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.

Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
being perfect. If you're due for an upgrade, definitely worth checking out!
"""
structure_llm = llm.with_structured_output(MobileReview)
output = structure_llm.invoke(review_text)
output

MobileReview(phone_model='Galaxy S21', rating=4.0, pros=['gorgeous screen', 'insane camera', 'solid battery life'], cons=['pricey', 'ditching charger', 'Bixby button issues'], summary='Solid phone with a few quirks')

In [48]:
output.model_dump()

{'phone_model': 'Galaxy S21',
 'rating': 4.0,
 'pros': ['gorgeous screen', 'insane camera', 'solid battery life'],
 'cons': ['pricey', 'ditching charger', 'Bixby button issues'],
 'summary': 'Solid phone with a few quirks'}

# PRompt template

In [68]:
from langchain.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")
prompt.invoke({"topic": "programming"})

ChatPromptValue(messages=[HumanMessage(content='Tell me a short joke about programming', additional_kwargs={}, response_metadata={})])

In [70]:
chain= prompt | llm | StrOutputParser()
ans= chain.invoke({"topic":"dogs"})
print(ans)

Why did the dog go to the vet? 

Because he was feeling a little ruff.


# LLM Message

In [75]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

sys_msg= SystemMessage(content="you an funny guy who raise the fustrations.")
human_msg= HumanMessage(content="I am feeling nervous today.")
llm.invoke([sys_msg, human_msg])

AIMessage(content="You think you're nervous? I'm feeling anxious just thinking about all the things that could go wrong in life. Like, have you ever tried to make toast? It's a miracle in itself, but what if the bread gets stuck? Or the toaster catches on fire? No, no, I'm good. I think I'll just stick to eating plain air for now.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 52, 'total_tokens': 129, 'completion_time': 2.8086786630000002, 'prompt_time': 0.217121207, 'queue_time': 0.122369015, 'total_time': 3.02579987}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_9cb648b966', 'finish_reason': 'stop', 'logprobs': None}, id='run-2b3a1427-14e8-42c1-a610-4a2864e639a7-0', usage_metadata={'input_tokens': 52, 'output_tokens': 77, 'total_tokens': 129})

In [78]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that tells jokes."),
    ("human", "Tell me about {topic}")
])
prompt_value = template.invoke(
    {
        "topic": "programming"
    }
)
prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that tells jokes.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about programming', additional_kwargs={}, response_metadata={})])

In [79]:
llm.invoke(prompt_value)

AIMessage(content='Programming! It\'s like trying to find the perfect recipe – you need the right ingredients, the right instructions, and a dash of creativity. Speaking of which, here\'s a programming-themed joke:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs.\n\nNow, about programming itself:\n\nProgramming is the process of designing, writing, testing, and maintaining the instructions that a computer follows to perform a specific task. It\'s like writing a recipe book for the computer, but instead of cooking a cake, you\'re creating a software or an app.\n\nThere are many programming languages, each with its own unique characteristics and uses. Some popular ones include:\n\n1. Python: Known for its simplicity and versatility, Python is often used for web development, data analysis, and machine learning.\n2. JavaScript: Used for client-side scripting in web development, JavaScript is a powerful tool for creating interactive web pages and applications.\n3. Jav

In [87]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [85]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

docx_loader = Docx2txtLoader("./docs/GreenGrow Innovations_ Company History.docx")
documents = docx_loader.load()

splits= text_splitter.split_documents(documents)
splits

[Document(metadata={'source': './docs/GreenGrow Innovations_ Company History.docx'}, page_content='GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors to analy

In [96]:
from typing import List, Literal
def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

##Load the documents from folder
folder_path = "./docs"
documents = load_documents(folder_path)

print(f"Loaded {len(documents)} documents from the folder.")
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks.")

Loaded 19 documents from the folder.
Split the documents into 57 chunks.


In [8]:
embedding_model= GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')
embedding_model.embed_query("Hi")

[0.03310242295265198,
 0.02527472749352455,
 -0.0730745866894722,
 -0.006820999551564455,
 -0.004950474016368389,
 0.022163614630699158,
 0.07135909050703049,
 0.041716188192367554,
 0.028002023696899414,
 0.05275154486298561,
 -0.08173676580190659,
 0.04290860891342163,
 0.04718734696507454,
 0.015800779685378075,
 -0.06697343289852142,
 -0.04090011492371559,
 -0.026385167613625526,
 0.024090725928544998,
 -0.0760846808552742,
 -0.008802172727882862,
 -0.02122870646417141,
 0.01640634424984455,
 -0.00818869099020958,
 -0.024566518142819405,
 -0.024494273588061333,
 0.042749661952257156,
 0.007968702353537083,
 0.018405139446258545,
 0.0004703607701230794,
 -0.029696231707930565,
 -0.012846840545535088,
 0.06671243160963058,
 0.01923949271440506,
 -0.020802674815058708,
 0.030476758256554604,
 0.022702811285853386,
 -0.05111874267458916,
 0.060959819704294205,
 0.02875923179090023,
 -0.1051102951169014,
 -0.010511391796171665,
 0.04115733131766319,
 -0.023653650656342506,
 -0.015645507

In [7]:
embedding_model= GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')

document_embeddings = embedding_model.embed_documents([split.page_content for split in splits])

print(f"Created embeddings for {len(document_embeddings)} document chunks.")

NameError: name 'splits' is not defined

# create ChromaDB

In [105]:
from langchain_chroma import Chroma
collection_name= "my_rag"

vectorstore = Chroma.from_documents(collection_name=collection_name, documents=splits, embedding=embedding_model, persist_directory="./chroma_db")

print("Vector store created and persisted to './chroma_db'")

Vector store created and persisted to './chroma_db'


In [107]:
# 5. Perform similarity search\\\\\\\\\\\\\\

query = "Who are the author of attention all you need paper?"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content: {result.page_content}")
    print()


Top 2 most relevant chunks for the query: 'Who are the author of attention all you need paper?'

Result 1:
Source: ./docs\transformer_paper.pdf
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We p

In [108]:
from langchain_core.prompts import ChatPromptTemplate
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(template)

In [110]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [113]:
from langchain.schema.runnable import RunnablePassthrough

rag_chain= (
    {"context":retriever, 'question': RunnablePassthrough()} |
    prompt
    )

rag_chain.invoke("What is self attention?")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\n[Document(id=\'a293c7f2-b5fd-4906-9bad-4b33a816d20e\', metadata={\'page\': 1, \'page_label\': \'2\', \'source\': \'./docs\\\\transformer_paper.pdf\'}, page_content=\'in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes\\nit more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is\\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\\ndescribed in section 3.2.\\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\\nused successfully in a variety of tasks including reading comprehension, abstractive summa

In [114]:
def docs2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [115]:
rag_chain= (
    {"context":retriever | docs2str, 'question': RunnablePassthrough()} |
    prompt
    )

rag_chain.invoke("What is self attention?")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\nin the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes\nit more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\ntextual entailment and learning task-independent sentence representations [4, 27, 28, 22].\nEnd-to-end memory networks are based on a recurrent attention mechanis

In [117]:
rag_chain = (
    {"context": retriever | docs2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
question = "Who written the Attenetion all you need paper"
response = rag_chain.invoke(question)
print(response)

Ashish Vaswani∗, Noam Shazeer∗, Niki Parmar∗, Jakob Uszkoreit∗, Llion Jones∗, Aidan N. Gomez∗, Łukasz Kaiser∗, and Illia Polosukhin∗.


# Conversational Rag

In [119]:

# Example conversation
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

In [120]:
chat_history

[HumanMessage(content='Who written the Attenetion all you need paper', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Ashish Vaswani∗, Noam Shazeer∗, Niki Parmar∗, Jakob Uszkoreit∗, Llion Jones∗, Aidan N. Gomez∗, Łukasz Kaiser∗, and Illia Polosukhin∗.', additional_kwargs={}, response_metadata={})]

In [136]:
from langchain_core.prompts import MessagesPlaceholder


contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke({"input":"Where it is headquartered?", "chat_history":chat_history})

'The authors, Ashish Vaswani et al, did not mention the headquarters in the paper.'

In [140]:
from langchain.chains import create_history_aware_retriever

history_aware_retriever= create_history_aware_retriever(llm, retriever, contextualize_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B661D8FF90>, search_kwargs={'k': 2}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(

In [141]:
history_aware_retriever.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

[Document(id='884fa392-5638-4b37-b9f3-21668b27bcb3', metadata={'source': './docs\\Company_ TechWave Innovations.docx'}, page_content='Company: TechWave Innovations\n\nHeadquarters: TechWave Innovations is headquartered in San Francisco, California, USA. As a leader in cutting-edge AI and machine learning solutions, the company thrives in the heart of Silicon Valley, benefiting from its proximity to tech giants and a dynamic startup ecosystem. With its headquarters in this global technology hub, TechWave Innovations has access to top talent and a vast network of innovation-driven enterprises.'),
 Document(id='5789c75d-631f-42c2-a30f-dd5e914ff0c3', metadata={'source': './docs\\Company_ QuantumNext Systems.docx'}, page_content='Company: QuantumNext Systems\n\nHeadquarters: QuantumNext Systems is headquartered in Bangalore, Karnataka, India. The company, specializing in quantum computing and advanced data processing, is situated in the bustling tech metropolis of Bangalore, often referred 

In [142]:

retriever.invoke("Where it is headquartered?")

[Document(id='884fa392-5638-4b37-b9f3-21668b27bcb3', metadata={'source': './docs\\Company_ TechWave Innovations.docx'}, page_content='Company: TechWave Innovations\n\nHeadquarters: TechWave Innovations is headquartered in San Francisco, California, USA. As a leader in cutting-edge AI and machine learning solutions, the company thrives in the heart of Silicon Valley, benefiting from its proximity to tech giants and a dynamic startup ecosystem. With its headquarters in this global technology hub, TechWave Innovations has access to top talent and a vast network of innovation-driven enterprises.'),
 Document(id='5789c75d-631f-42c2-a30f-dd5e914ff0c3', metadata={'source': './docs\\Company_ QuantumNext Systems.docx'}, page_content='Company: QuantumNext Systems\n\nHeadquarters: QuantumNext Systems is headquartered in Bangalore, Karnataka, India. The company, specializing in quantum computing and advanced data processing, is situated in the bustling tech metropolis of Bangalore, often referred 

In [143]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the following context to answer the user's question."),
    #  ("system", "Tell me joke on Programming"),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [144]:

rag_chain.invoke({"input": "Where it is headquartered?", "chat_history":chat_history})

{'input': 'Where it is headquartered?',
 'chat_history': [HumanMessage(content='Who written the Attenetion all you need paper', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Ashish Vaswani∗, Noam Shazeer∗, Niki Parmar∗, Jakob Uszkoreit∗, Llion Jones∗, Aidan N. Gomez∗, Łukasz Kaiser∗, and Illia Polosukhin∗.', additional_kwargs={}, response_metadata={})],
 'context': [Document(id='57d19eb2-7ad7-4bad-b0e5-9b7df33a842c', metadata={'page': 0, 'page_label': '1', 'source': './docs\\transformer_paper.pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. G

# Multi user chatbot

In [154]:
import sqlite3
from datetime import datetime

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS application_logs
                    (id INTEGER PRIMARY KEY AUTOINCREMENT,
                     session_id TEXT,
                     user_query TEXT,
                     gpt_response TEXT,
                     model TEXT,
                     created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute('INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?, ?, ?, ?)',
                 (session_id, user_query, gpt_response, model))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at', (session_id,))
    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])
    conn.close()
    return messages

# Initialize the database
create_application_logs()

In [155]:
import uuid
session_id = str(uuid.uuid4())
chat_history = get_chat_history(session_id)
print(chat_history)
question1 = "When was GreenGrow Innovations founded?"
answer1 = rag_chain.invoke({"input": question1, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question1, answer1, "gpt-4-o-mini")
print(f"Human: {question1}")
print(f"AI: {answer1}\n")

[]
Human: When was GreenGrow Innovations founded?
AI: GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming.



In [156]:
question2 = "Where it is headquartered?"
chat_history = get_chat_history(session_id)
print(chat_history)
answer2 = rag_chain.invoke({"input": question2, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gpt-3.5-turbo")
print(f"Human: {question2}")
print(f"AI: {answer2}\n")

[{'role': 'human', 'content': 'When was GreenGrow Innovations founded?'}, {'role': 'ai', 'content': 'GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming.'}]
Human: Where it is headquartered?
AI: The context does not provide information about GreenGrow's headquarters location. However, it does mention that the company has offices in California and Iowa, and its early operations were based in Portland, Oregon.



In [157]:
session_id = str(uuid.uuid4())
question = "What is GreenGrow"
chat_history = get_chat_history(session_id)
print(chat_history)
answer = rag_chain.invoke({"input": question, "chat_history":chat_history})['answer']
insert_application_logs(session_id, question, answer, "gpt-3.5-turbo")
print(f"Human: {question}")
print(f"AI: {answer}\n")

[]
Human: What is GreenGrow
AI: GreenGrow Innovations is a company that specializes in developing sustainable agricultural technologies. It was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for promoting environmentally friendly and efficient farming practices.

GreenGrow is known for its breakthrough EcoHarvest System, introduced in 2018, which combines smart irrigation, soil monitoring, and automated harvesting techniques. The company has expanded its operations to include offices in California and Iowa, and it continues to focus on developing innovative solutions for the agricultural industry, such as vertical farming, drought-resistant crop development, and AI-powered farm management systems.

GreenGrow aims to make farming more sustainable, efficient, and environmentally friendly, and it regularly partners with universities and research institutions to advance the field of agricultural technology. The company also hosts annual confe

In [130]:
from langchain.schema.runnable import RunnablePassthrough

# Create a RunnablePassthrough instance
passthrough = RunnablePassthrough()

# Input data
input_data = {"key": "value"}

# Run it through the passthrough, which just returns the same data
output_data = passthrough.invoke(input_data)

# Print the result, which will be the same as input_data
print(output_data)  # Output: {"key": "value"}


{'key': 'value'}
